# A Practical Hybrid Diffusion–GAN Approach to Image Super-Resolution on CPU

This notebook implements and documents the experiments described in the report:

> **"A Practical Hybrid Diffusion--GAN Approach to Image Super-Resolution on CPU"**

## Goals

- Implement a **CPU-friendly hybrid SR model** combining:
  - A lightweight GAN-inspired base SR network
  - A diffusion-style, edge-conditioned refiner
- Respect strict **latency constraints**:
  - Quality-focused hybrid: **8–20 seconds** per image
  - High-quality refiner variant: up to **120 seconds** per image
- Compare:
  - Base SR vs Hybrid SR
  - Fast vs Quality configurations
- Reflect on:
  - How this hybrid relates to diffusion and GAN SR
  - Whether diffusion “beats” GAN in this practical CPU setting


In [ ]:
import os
import glob
import time
import random
from typing import Tuple

import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
import torchvision.transforms.functional as TF


In [ ]:
# Device: CPU-only by design
device = torch.device("cpu")
torch.set_num_threads(2)  # Tune if needed

# Paths (update to match your system)
IMAGENET_DIR = r"C:\Users\book\.cache\kagglehub\datasets\deeptrial\miniimagenet\versions\2"
ORIGINAL_IMAGE_PATH = "original.png"

# Where to save refiner weights
QUALITY_REFINER_PATH = "quality_hybrid_refiner.pth"      # 8–20s target
HIGH_QUALITY_REFINER_PATH = "high_quality_refiner.pth"  # ≤120s target


## 1. Model Architecture

We implement three main components:

1. **ImprovedSRModel**  
   - Lightweight base SR using bicubic upsampling + shallow CNN refinement.

2. **BalancedRefiner / EnhancedRefiner**  
   - Diffusion-style refiner, conditioned on edges, predicting a residual.

3. **QualityHybridSRModel / HighQualityHybridSRModel**  
   - Wrapper that:
     - Runs base SR
     - Computes edge maps
     - Applies refiner
     - Enforces latency constraints in the forward pass


In [ ]:
class ImprovedSRModel(nn.Module):
    """
    Lightweight base SR model:
    - Bicubic upsampling by factor 4
    - 3 conv layers with residual refinement
    """
    def __init__(self, scale: int = 4, base_channels: int = 32):
        super().__init__()
        self.scale = scale

        self.conv1 = nn.Conv2d(3, base_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(base_channels, base_channels, 3, padding=1)
        self.conv3 = nn.Conv2d(base_channels, 3, 3, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

        # Control the strength of refinement
        self.refine_scale = 0.2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W)
        up = torch.nn.functional.interpolate(
            x, scale_factor=self.scale,
            mode="bicubic", align_corners=False
        )
        h = self.act(self.conv1(up))
        h = self.act(self.conv2(h))
        h = self.conv3(h)
        out = torch.clamp(up + self.refine_scale * h, 0.0, 1.0)
        return out

Balanced Refiner (Quality 8–20s)

In [ ]:
class BalancedRefiner(nn.Module):
    """
    Edge-conditioned refiner:
    - Input: base SR image + edge map (4 channels)
    - Output: residual refinement
    """
    def __init__(self, base_channels: int = 32, skip_scale: float = 0.2):
        super().__init__()
        in_ch = 3 + 1  # RGB + edge
        self.skip_scale = skip_scale

        self.conv1 = nn.Conv2d(in_ch, base_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(base_channels, base_channels, 3, padding=1)
        self.conv3 = nn.Conv2d(base_channels, 3, 3, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, img: torch.Tensor, edge: torch.Tensor) -> torch.Tensor:
        # Concatenate image and edge map: (B, 4, H, W)
        x = torch.cat([img, edge], dim=1)
        h = self.act(self.conv1(x))
        h = self.act(self.conv2(h))
        h = self.conv3(h)
        return self.skip_scale * h


Enhanced Refiner (High-Quality ≤120s)

In [ ]:
class EnhancedRefiner(nn.Module):
    """
    Higher-capacity refiner:
    - More channels, more layers, intended for up to ~2 minutes per image.
    """
    def __init__(self, base_channels: int = 64, skip_scale: float = 0.25):
        super().__init__()
        in_ch = 3 + 1  # RGB + edge
        self.skip_scale = skip_scale

        self.conv1 = nn.Conv2d(in_ch, base_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(base_channels, base_channels, 3, padding=1)
        self.conv3 = nn.Conv2d(base_channels, base_channels, 3, padding=1)
        self.conv4 = nn.Conv2d(base_channels, 3, 3, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, img: torch.Tensor, edge: torch.Tensor) -> torch.Tensor:
        x = torch.cat([img, edge], dim=1)
        h = self.act(self.conv1(x))
        h = self.act(self.conv2(h))
        h = self.act(self.conv3(h))
        h = self.conv4(h)
        return self.skip_scale * h


Edge Map Utility 

In [ ]:
def sobel_edge_map(x: torch.Tensor) -> torch.Tensor:
    """
    Compute Sobel edge magnitude for a batch of RGB images in [0,1].
    Returns a 1-channel edge map with the same H, W.
    """
    # Convert to grayscale
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    gray = 0.299 * r + 0.587 * g + 0.114 * b

    sobel_x = torch.tensor([[1, 0, -1],
                            [2, 0, -2],
                            [1, 0, -1]], dtype=torch.float32, device=gray.device).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[1, 2, 1],
                            [0, 0, 0],
                            [-1, -2, -1]], dtype=torch.float32, device=gray.device).view(1, 1, 3, 3)

    gx = torch.nn.functional.conv2d(gray, sobel_x, padding=1)
    gy = torch.nn.functional.conv2d(gray, sobel_y, padding=1)

    mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)
    mag = mag / (mag.max() + 1e-8)
    return mag


Hybrid Models (Quality vs High Quality)

In [ ]:
class QualityHybridSRModel(nn.Module):
    """
    Hybrid SR model (8–20 second target).
    - Base SR + BalancedRefiner
    - Latency-aware forward pass.
    """
    def __init__(self, base_sr: nn.Module, refiner: nn.Module, max_total_time: float = 20.0):
        super().__init__()
        self.base_sr = base_sr
        self.refiner = refiner
        self.max_total_time = max_total_time

    def forward(self, lr: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        start_time = time.time()

        # Base SR
        base_sr = self.base_sr(lr)
        base_time = time.time() - start_time

        # If we already used almost all budget, return base only
        if base_time > self.max_total_time * 0.8:
            return base_sr, base_sr

        # Edge map
        edge = sobel_edge_map(base_sr)

        # Refiner
        residual = self.refiner(base_sr, edge)
        hybrid = torch.clamp(base_sr + residual, 0.0, 1.0)

        total_time = time.time() - start_time
        if total_time > self.max_total_time:
            print(f"Hybrid forward took {total_time:.2f}s (over {self.max_total_time}s)")
        return hybrid, base_sr


class HighQualityHybridSRModel(nn.Module):
    """
    High-quality hybrid with larger refiner and max ~120s target.
    """
    def __init__(self, base_sr: nn.Module, refiner: nn.Module, max_total_time: float = 120.0):
        super().__init__()
        self.base_sr = base_sr
        self.refiner = refiner
        self.max_total_time = max_total_time

    def forward(self, lr: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        start_time = time.time()

        base_sr = self.base_sr(lr)
        base_time = time.time() - start_time

        edge = sobel_edge_map(base_sr)
        residual = self.refiner(base_sr, edge)
        hybrid = torch.clamp(base_sr + residual, 0.0, 1.0)

        total_time = time.time() - start_time
        if total_time > self.max_total_time:
            print(f"High-quality hybrid took {total_time:.1f}s (> {self.max_total_time:.0f}s)")

        return hybrid, base_sr


## 2. Dataset and Preprocessing

We train on a small subset of ImageNet-like images:

- Random HR patches of size `(4 × patch_size)²` (e.g. 192×192 or 384×384)
- Generate LR patches via bicubic downscaling (patch_size × patch_size)
- Basic augmentations:
  - Random horizontal flips
  - Optional rotations


In [ ]:
class QualityImageNetSRDataset(Dataset):
    """
    Dataset for quality-focused hybrid (8–20 second target).
    """
    def __init__(self, imagenet_dir: str, patch_size: int = 48, scale: int = 4, max_images: int = 40):
        super().__init__()
        paths = sorted(glob.glob(os.path.join(imagenet_dir, "**", "*.JPEG"), recursive=True))
        if len(paths) == 0:
            paths = sorted(glob.glob(os.path.join(imagenet_dir, "**", "*.jpg"), recursive=True)) + \
                    sorted(glob.glob(os.path.join(imagenet_dir, "**", "*.png"), recursive=True))
        self.paths = paths[:max_images]
        self.patch_size = patch_size
        self.scale = scale

    def __len__(self):
        # Reuse each image many times with random crops
        return len(self.paths) * 20

    def __getitem__(self, idx):
        img_idx = idx % len(self.paths)
        hr_img = Image.open(self.paths[img_idx]).convert("RGB")
        hr_tensor = TF.to_tensor(hr_img)  # (C, H, W)

        C, H, W = hr_tensor.shape
        ps_hr = self.patch_size * self.scale

        if H > ps_hr and W > ps_hr:
            x = random.randint(0, W - ps_hr)
            y = random.randint(0, H - ps_hr)
            hr_crop = hr_tensor[:, y:y+ps_hr, x:x+ps_hr]
        else:
            hr_crop = TF.resize(hr_tensor, [ps_hr, ps_hr])

        lr_crop = TF.resize(
            hr_crop, [self.patch_size, self.patch_size],
            interpolation=transforms.InterpolationMode.BICUBIC
        )

        if random.random() > 0.5:
            lr_crop = TF.hflip(lr_crop)
            hr_crop = TF.hflip(hr_crop)

        return lr_crop, hr_crop


High-Quality Dataset

In [ ]:
class HighQualityImageNetSRDataset(Dataset):
    """
    Larger patches and more augmentation for high-quality refiner (≤120s).
    """
    def __init__(self, imagenet_dir: str, patch_size: int = 96, scale: int = 4, max_images: int = 40):
        super().__init__()
        paths = sorted(glob.glob(os.path.join(imagenet_dir, "**", "*.JPEG"), recursive=True))
        if len(paths) == 0:
            paths = sorted(glob.glob(os.path.join(imagenet_dir, "**", "*.jpg"), recursive=True)) + \
                    sorted(glob.glob(os.path.join(imagenet_dir, "**", "*.png"), recursive=True))
        self.paths = paths[:max_images]
        self.patch_size = patch_size
        self.scale = scale

    def __len__(self):
        return len(self.paths) * 20

    def __getitem__(self, idx):
        img_idx = idx % len(self.paths)
        hr_img = Image.open(self.paths[img_idx]).convert("RGB")
        hr_tensor = TF.to_tensor(hr_img)
        C, H, W = hr_tensor.shape

        ps_hr = self.patch_size * self.scale
        if H > ps_hr and W > ps_hr:
            x = random.randint(0, W - ps_hr)
            y = random.randint(0, H - ps_hr)
            hr_crop = hr_tensor[:, y:y+ps_hr, x:x+ps_hr]
        else:
            hr_crop = TF.resize(hr_tensor, [ps_hr, ps_hr])

        lr_crop = TF.resize(
            hr_crop, [self.patch_size, self.patch_size],
            interpolation=transforms.InterpolationMode.BICUBIC
        )

        if random.random() > 0.5:
            lr_crop = TF.hflip(lr_crop)
            hr_crop = TF.hflip(hr_crop)

        if random.random() > 0.7:
            angle = random.choice([0, 90, 180, 270])
            lr_crop = TF.rotate(lr_crop, angle)
            hr_crop = TF.rotate(hr_crop, angle)

        return lr_crop, hr_crop


Loss & Metric Utilities

In [ ]:
criterion_l1 = nn.L1Loss()
criterion_mse = nn.MSELoss()

def combined_loss(pred, target):
    return criterion_l1(pred, target) + 0.5 * criterion_mse(pred, target)

def psnr(mse: float, max_val: float = 1.0) -> float:
    return 10.0 * np.log10((max_val ** 2) / (mse + 1e-8))

def ssim_gray(img1: np.ndarray, img2: np.ndarray) -> float:
    """
    Simplified SSIM on grayscale images.
    img1, img2: [H, W] in [0,1]
    """
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    mu1 = cv2.GaussianBlur(img1, (11, 11), 1.5)
    mu2 = cv2.GaussianBlur(img2, (11, 11), 1.5)
    sigma1_sq = cv2.GaussianBlur(img1**2, (11, 11), 1.5) - mu1**2
    sigma2_sq = cv2.GaussianBlur(img2**2, (11, 11), 1.5) - mu2**2
    sigma12 = cv2.GaussianBlur(img1*img2, (11, 11), 1.5) - mu1*mu2

    ssim_map = ((2*mu1*mu2 + C1) * (2*sigma12 + C2)) / ((mu1**2 + mu2**2 + C1) *
                                                         (sigma1_sq + sigma2_sq + C2))
    return float(ssim_map.mean())

def sharpness_variance_of_laplacian(gray: np.ndarray) -> float:
    lap = cv2.Laplacian((gray * 255).astype(np.uint8), cv2.CV_64F)
    return float(lap.var())


Training: Refiner-Only (QualityHybrid, 8–20s)

In [ ]:
def train_refiner_only(
    imagenet_dir: str,
    out_refiner_path: str = QUALITY_REFINER_PATH,
    batch_size: int = 2,
    patch_size: int = 48,
    epochs: int = 5,
    lr_rate: float = 5e-4,
    max_images: int = 40,
):
    print("🚀 REFINER-ONLY HYBRID SR MODEL TRAINING (8–20s target)")
    dataset = QualityImageNetSRDataset(imagenet_dir, patch_size=patch_size, scale=4, max_images=max_images)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    base_sr_model = ImprovedSRModel(scale=4, base_channels=32).to(device)
    for p in base_sr_model.parameters():
        p.requires_grad = False
    base_sr_model.eval()

    refiner = BalancedRefiner(base_channels=32).to(device)
    hybrid_model = QualityHybridSRModel(base_sr_model, refiner, max_total_time=20.0).to(device)

    optimizer = optim.AdamW(refiner.parameters(), lr=lr_rate, weight_decay=1e-4)

    print(f"Dataset patches: {len(dataset)} from {len(dataset.paths)} images")
    for epoch in range(1, epochs + 1):
        hybrid_model.train()
        running_loss = 0.0

        for i, (lr_img, hr_img) in enumerate(loader):
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            optimizer.zero_grad()
            hybrid_sr, base_sr = hybrid_model(lr_img)
            loss = combined_loss(hybrid_sr, hr_img)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(refiner.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
        print(f"[Epoch {epoch}/{epochs}] Loss: {running_loss / len(loader):.4f}")

    torch.save(refiner.state_dict(), out_refiner_path)
    print(f"✅ Saved quality refiner weights to {out_refiner_path}")


Training: High-Quality Refiner (≤120s)

In [ ]:
def train_high_quality_refiner(
    imagenet_dir: str,
    out_refiner_path: str = HIGH_QUALITY_REFINER_PATH,
    batch_size: int = 2,
    patch_size: int = 96,
    epochs: int = 8,
    lr_rate: float = 3e-4,
    max_images: int = 40,
):
    print("HIGH-QUALITY REFINER-ONLY SR MODEL TRAINING (≤120s target)")
    dataset = HighQualityImageNetSRDataset(imagenet_dir, patch_size=patch_size, scale=4, max_images=max_images)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    base_sr_model = ImprovedSRModel(scale=4, base_channels=32).to(device)
    for p in base_sr_model.parameters():
        p.requires_grad = False
    base_sr_model.eval()

    refiner = EnhancedRefiner(base_channels=64).to(device)
    hybrid_model = HighQualityHybridSRModel(base_sr_model, refiner, max_total_time=120.0).to(device)

    optimizer = optim.AdamW(refiner.parameters(), lr=lr_rate, weight_decay=1e-4)

    print(f"Dataset patches: {len(dataset)} from {len(dataset.paths)} images")
    for epoch in range(1, epochs + 1):
        hybrid_model.train()
        running_loss = 0.0

        for i, (lr_img, hr_img) in enumerate(loader):
            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            optimizer.zero_grad()
            hybrid_sr, base_sr = hybrid_model(lr_img)
            loss = combined_loss(hybrid_sr, hr_img)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(refiner.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
        print(f"[Epoch {epoch}/{epochs}] Loss: {running_loss / len(loader):.4f}")

    torch.save(refiner.state_dict(), out_refiner_path)
    print(f"Saved high-quality refiner weights to {out_refiner_path}")


Inference on original.png + Metrics

In [ ]:
def load_image_as_tensor(path: str) -> torch.Tensor:
    img = Image.open(path).convert("RGB")
    tensor = TF.to_tensor(img).unsqueeze(0)
    return tensor

def tensor_to_numpy_img(t: torch.Tensor) -> np.ndarray:
    # t: (1, 3, H, W) in [0,1]
    img = t.squeeze(0).permute(1, 2, 0).cpu().numpy()
    return np.clip(img, 0.0, 1.0)

def evaluate_model_on_original(
    model: nn.Module,
    description: str,
    lr_scale_factor: int = 4,
):
    print(f"\n===== Evaluation: {description} =====")
    assert os.path.exists(ORIGINAL_IMAGE_PATH), f"{ORIGINAL_IMAGE_PATH} not found"
    # Treat original as LR for demonstration
    lr = load_image_as_tensor(ORIGINAL_IMAGE_PATH).to(device)

    start_time = time.time()
    hybrid_sr, base_sr = model(lr)
    total_time = time.time() - start_time
    print(f"Runtime: {total_time:.2f} seconds")

    # Resize original to match SR size for metric comparison
    H, W = lr.shape[2], lr.shape[3]
    target_H, target_W = H * lr_scale_factor, W * lr_scale_factor
    original_resized = TF.resize(
        TF.to_tensor(Image.open(ORIGINAL_IMAGE_PATH).convert("RGB")),
        [target_H, target_W],
        interpolation=transforms.InterpolationMode.BICUBIC
    ).permute(1, 2, 0).numpy()

    base_np = tensor_to_numpy_img(base_sr)
    hybrid_np = tensor_to_numpy_img(hybrid_sr)

    # Convert to [0,1] grayscale
    def to_gray(x):
        return cv2.cvtColor((x * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY) / 255.0

    orig_gray = to_gray(original_resized)
    base_gray = to_gray(base_np)
    hybrid_gray = to_gray(hybrid_np)

    mse_base = np.mean((base_np - original_resized) ** 2)
    mse_hybrid = np.mean((hybrid_np - original_resized) ** 2)

    psnr_base = psnr(mse_base)
    psnr_hybrid = psnr(mse_hybrid)

    ssim_base = ssim_gray(orig_gray, base_gray)
    ssim_hybrid = ssim_gray(orig_gray, hybrid_gray)

    sharp_base = sharpness_variance_of_laplacian(base_gray)
    sharp_hybrid = sharpness_variance_of_laplacian(hybrid_gray)

    print(f"PSNR (Base):   {psnr_base:.2f} dB")
    print(f"PSNR (Hybrid): {psnr_hybrid:.2f} dB")
    print(f"SSIM (Base):   {ssim_base:.3f}")
    print(f"SSIM (Hybrid): {ssim_hybrid:.3f}")
    print(f"Sharpness (Base):   {sharp_base:.1f}")
    print(f"Sharpness (Hybrid): {sharp_hybrid:.1f}")

    # Save outputs
    os.makedirs("results", exist_ok=True)
    Image.fromarray((base_np * 255).astype(np.uint8)).save("results/base_sr_original.png")
    Image.fromarray((hybrid_np * 255).astype(np.uint8)).save("results/hybrid_sr_original.png")
    print("Saved: results/base_sr_original.png, results/hybrid_sr_original.png")


## 3. Reflection: Hybrid vs Pure GAN/Diffusion

In this notebook, we implemented:
- A **GAN-inspired base SR** (ImprovedSRModel)
- A **diffusion-style edge-conditioned refiner** (BalancedRefiner / EnhancedRefiner)
- Two practical regimes:
  - **QualityHybridSRModel**: 8–20 s target
  - **HighQualityHybridSRModel**: ≤120 s target

This mirrors the report:
> "A Practical Hybrid Diffusion--GAN Approach to Image Super-Resolution on CPU"

Key points:
- We borrow *stability and structure* from diffusion (via single-step denoising-like residuals).
- We borrow *sharpness and detail* from GAN-style SR backbones.
- Our design is explicitly **CPU-aware** and **latency-constrained**, unlike most academic hybrids.

For a full theoretical comparison (Diffusion vs GAN), see the accompanying written report.


Main” Cell to Run Training + Evaluation (Code)

In [ ]:
if __name__ == "__main__":
    # 1) Train 8–20s quality refiner
    # train_refiner_only(IMAGENET_DIR)

    # 2) Train ≤120s high-quality refiner
    # train_high_quality_refiner(IMAGENET_DIR)

    # 3) Load base + quality refiner and test
    base_model = ImprovedSRModel(scale=4, base_channels=32).to(device)
    quality_refiner = BalancedRefiner(base_channels=32).to(device)
    if os.path.exists(QUALITY_REFINER_PATH):
        quality_refiner.load_state_dict(torch.load(QUALITY_REFINER_PATH, map_location=device))
        print("Loaded quality refiner weights.")
    quality_hybrid = QualityHybridSRModel(base_model, quality_refiner, max_total_time=20.0).to(device)

    evaluate_model_on_original(quality_hybrid, "QualityHybridSRModel (8–20s)")

    # 4) Load high-quality refiner and test
    hq_refiner = EnhancedRefiner(base_channels=64).to(device)
    if os.path.exists(HIGH_QUALITY_REFINER_PATH):
        hq_refiner.load_state_dict(torch.load(HIGH_QUALITY_REFINER_PATH, map_location=device))
        print("Loaded high-quality refiner weights.")
    hq_hybrid = HighQualityHybridSRModel(base_model, hq_refiner, max_total_time=120.0).to(device)

    evaluate_model_on_original(hq_hybrid, "HighQualityHybridSRModel (≤120s)")
